# NDI Task 4, Stage 0: learnability check on the real data

**Task 3 review and Task 4 execution memo, Section 2.** Run this before anything else in Task 4 (about 20-25 minutes, **Runtime > Run all**). No tag is needed: it is a diagnostic, labelled exploratory, and runs the Task 3 code under tag `v1.4-task3`. G-0 asks once for the read-only token.

- Data cells are the v1.4 cells, unchanged. V12-1, V12-2 and T3-1 are the Task 3 cells, unchanged.
- **S0-0** settings checked against tags `v1.4` and `v1.4-task3`; Stage 0 config and provenance.
- **S0-2** one diagnostic arm, *learned persistence*: personalised Z's model (fold z-scoring, then Ridge with the same RIDGE_ALPHA) given a single feature, today's value of the target quantity, under the exact v1.4 / Task 3 fold protocol. Self-derived: today's causal index; raw-feature: today's causal raw composite; next-day mood / stress: today's W1-W4 value (row dropped if missing). Checks: the same scored days as the archived runs, and no-model persistence reproduces the archived Task 3 values exactly.
- **S0-3** table and the fixed decision: **PROCEED** if learned persistence is at least half the no-model value on every target; otherwise **STOP** and send the per-participant values.

- **S0-4** (after STOP) splits each learned prediction into slope x today's value and intercept. **S0-5 = Stage 0b** (Stage 0 review memo, Section 1): anomaly re-scoring, rho_anom = Spearman(f - a, y - a), a = the fold's training mean of the target, for learned and no-model persistence on the same days; diagnostic, exploratory, gates nothing.

Writes only to `results_task4_stage0/`. Every archive is read, never written.

## Data (v1.4 cells, unchanged)

In [ ]:
# ================================================================
# CELL 1.0  CONFIGURATION  (edit paths to match your Drive)
# Brief Section 6: all parameters in one place, fixed seed
# ================================================================

#  Google Drive paths
PIPELINE_MODULE   = "/content/drive/MyDrive/ndi_project/ndi_pipeline.py"
DEPRESJON_DIR     = "/content/drive/MyDrive/ndi_project/depresjon-dataset/data"
PMDATA_DIR        = "/content/drive/MyDrive/ndi_project/pmdata"
PMDATA_OVERVIEW   = "/content/drive/MyDrive/ndi_project/pmdata/participant-overview.xlsx"
RESULTS_DIR       = "/content/drive/MyDrive/ndi_project/results"

#  Reproducibility seed (Brief Section 6)
RANDOM_SEED = 42

#  Mount Drive
from google.colab import drive
drive.mount("/content/drive")

import os, sys
os.makedirs(RESULTS_DIR, exist_ok=True)

# Add pipeline module to path
sys.path.insert(0, os.path.dirname(PIPELINE_MODULE))

print("Drive mounted.")
print("DEPRESJON dir exists:", os.path.isdir(DEPRESJON_DIR))
print("PMData    dir exists:", os.path.isdir(PMDATA_DIR))
print("Pipeline  exists    :", os.path.isfile(PIPELINE_MODULE))

In [ ]:
# ================================================================
# CELL 2  INSTALL DEPENDENCIES + IMPORTS
# ================================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["xgboost", "shap"]:
    install(pkg)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

import importlib.util, sys as _sys

spec = importlib.util.spec_from_file_location("ndi_pipeline", PIPELINE_MODULE)
ndi  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ndi)

print("ndi_pipeline loaded. Components available:")
for fn in ["neutrosophic_decompose","blocked_tscv","run_baselines",
           "lambda_grid_search","shap_attribution"]:
    print(f"  ndi.{fn}: {'OK' if hasattr(ndi, fn) else 'MISSING'}")

In [ ]:
# ================================================================
# CELL 3 -- LOAD DEPRESJON DATASET
# Structure: depresjon-dataset/data/condition/condition_1.csv ...
#            depresjon-dataset/data/control/control_1.csv ...
#            depresjon-dataset/data/scores.csv
# scores.csv: number=condition_1/condition_2/control_1 etc.
# ================================================================
import os, glob
import pandas as pd
import numpy as np

def load_depresjon(base_dir):
    scores_path = os.path.join(base_dir, "scores.csv")
    scores      = pd.read_csv(scores_path)
    scores.columns = [c.strip().lower() for c in scores.columns]
    print("scores.csv columns:", list(scores.columns))
    print(scores.head(3))

    # Build lookup: pid_string -> (madrs1, madrs2)
    madrs_lookup = {}
    for _, row in scores.iterrows():
        key = str(row["number"]).strip()
        m1  = float(row["madrs1"]) if str(row.get("madrs1","NA")).strip() not in ["NA","nan",""] else np.nan
        m2  = float(row["madrs2"]) if str(row.get("madrs2","NA")).strip() not in ["NA","nan",""] else np.nan
        madrs_lookup[key] = (m1, m2)

    participants = []

    for group in ["condition", "control"]:
        folder = os.path.join(base_dir, group)
        csvs   = sorted(glob.glob(os.path.join(folder, "*.csv")))
        print(f"\n{group}: {len(csvs)} files found")

        for csv_path in csvs:
            pid = os.path.splitext(os.path.basename(csv_path))[0]
            try:
                df = pd.read_csv(csv_path)
                df.columns = [c.strip().lower() for c in df.columns]

                if "timestamp" not in df.columns or "activity" not in df.columns:
                    print(f"  SKIP {pid}: columns={list(df.columns)}")
                    continue

                madrs1, madrs2 = madrs_lookup.get(pid, (np.nan, np.nan))
                madrs_mean     = np.nanmean([madrs1, madrs2])

                daily = ndi.extract_daily_features(df)

                if len(daily) < 5:
                    print(f"  SKIP {pid}: only {len(daily)} days")
                    continue

                participants.append(dict(
                    pid          = pid,
                    group        = group,
                    is_depressed = int(group == "condition"),
                    madrs1       = madrs1,
                    madrs2       = madrs2,
                    madrs_mean   = madrs_mean,
                    n_days       = len(daily),
                    days_df      = daily,
                ))
                print(f"  OK {pid}: {len(daily)} days  MADRS={madrs_mean:.0f}" if not np.isnan(madrs_mean)
                      else f"  OK {pid}: {len(daily)} days  MADRS=N/A")

            except Exception as e:
                print(f"  ERROR {pid}: {e}")

    return participants

depresjon_participants = load_depresjon(DEPRESJON_DIR)

condition_p = [p for p in depresjon_participants if p["group"] == "condition"]
control_p   = [p for p in depresjon_participants if p["group"] == "control"]
print("  NOTE: madrs_mean = NaN for control participants — expected, DEPRESJON administered MADRS to condition group only.")
print(f"\nLoaded total: {len(depresjon_participants)} participants")
print(f"  Condition (depressed): {len(condition_p)}")
print(f"  Control   (healthy)  : {len(control_p)}")
if condition_p:
    print(f"  Median days condition: {np.median([p['n_days'] for p in condition_p]):.0f}")
    valid_madrs = [p['madrs_mean'] for p in condition_p if not np.isnan(p['madrs_mean'])]
    if valid_madrs:
        print(f"  MADRS range: {min(valid_madrs):.0f} to {max(valid_madrs):.0f}")
if control_p:
    print(f"  Median days control  : {np.median([p['n_days'] for p in control_p]):.0f}")

In [ ]:
# ================================================================
# CELL 4  BUILD DEPRESJON FEATURE MATRIX
# FIX 2: neutro_df built here is FOR VISUALISATION ONLY.
# All CV metrics rebuild features per-fold in Cell 8.
# FIX 3: calls ndi.build_neutrosophic_features (module is
# authoritative). Inline decomposition deleted.
# ================================================================

FEAT_NAMES = ndi.DEPRESJON_FEATURE_NAMES
DOMAIN_MAP = ndi.DEPRESJON_DOMAIN_MAP

def build_participant_neutrosophic(p):
    """
    For one participant: extract feature matrix, apply neutrosophic
    decomposition using full-series train_mask (visualisation only).
    WARNING: Do NOT use neutro_df from this function in any CV result.
    All CV evaluation rebuilds features inside each fold (Cell 8).
    """
    df = p["days_df"].copy().reset_index(drop=True)
    for col in FEAT_NAMES:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = 0.0

    X = df[FEAT_NAMES].values
    n = len(X)

    # Full-series train_mask: all True — avoids the ValueError in
    # build_neutrosophic_features while making clear this is in-sample.
    # VISUALISATION ONLY — do not use for any headline metric.
    train_mask_full = np.ones(n, dtype=bool)
    neutro_df = ndi.build_neutrosophic_features(
        X, FEAT_NAMES, train_mask=train_mask_full)

    result = pd.concat([df[["date"]].reset_index(drop=True),
                        neutro_df.reset_index(drop=True)], axis=1)
    result["pid"]          = p["pid"]
    result["group"]        = p["group"]
    result["is_depressed"] = p["is_depressed"]
    result["madrs_mean"]   = p["madrs_mean"]
    for col in FEAT_NAMES:
        result[col] = df[col].values
    return result

print("Computing neutrosophic features (visualisation only)...")
neutro_frames = []
for p in depresjon_participants:
    try:
        nf = build_participant_neutrosophic(p)
        neutro_frames.append(nf)
        p["neutro_df"] = nf
    except Exception as e:
        print(f"  ERROR {p['pid']}: {e}")

depresjon_all = pd.concat(neutro_frames, ignore_index=True)
print(f"Total rows: {len(depresjon_all)}")
print("NOTE: neutro_df is for visualisation only. CV metrics use per-fold features.")

In [ ]:
# ================================================================
# CELL 5 -- LOAD PMDATA DATASET   [v1.4: two changes, marked 'v1.4' below]
# Actual structure:
#   pmdata/p01/fitbit/heart_rate.json
#   pmdata/p01/fitbit/resting_heart_rate.json
#   pmdata/p01/fitbit/sleep.json
#   pmdata/p01/fitbit/sleep_score.csv
#   pmdata/p01/fitbit/steps.json
#   pmdata/p01/fitbit/lightly_active_minutes.json
#   pmdata/p01/fitbit/moderately_active_minutes.json
#   pmdata/p01/fitbit/very_active_minutes.json
#   pmdata/p01/fitbit/sedentary_minutes.json
#   pmdata/p01/pmsys/wellness.csv    <- daily mood/wellness self-report
#   pmdata/p01/pmsys/srpe.csv
#   pmdata/participant-overview.xlsx <- max HR, age, gender per participant
# ================================================================
import os, glob, json as _json
import pandas as pd
import numpy as np
import openpyxl  # for reading xlsx

#  Load participant overview
try:
    overview = pd.read_excel(PMDATA_OVERVIEW, header=1)
    overview.columns = [str(c).strip().lower().replace(" ","_")
                        for c in overview.columns]
    overview = overview.dropna(subset=[overview.columns[0]])
    print("participant-overview columns:", list(overview.columns))
    print(overview.head(3))
except Exception as e:
    print(f"Could not load participant-overview: {e}")
    overview = pd.DataFrame()

#  Helper: load one JSON Fitbit file
def load_fitbit_json(path):
    """Load a Fitbit JSON file. Returns list of {dateTime, value} dicts."""
    try:
        with open(path) as f:
            data = _json.load(f)
        if isinstance(data, list):
            return data
        return []
    except Exception:
        return []

def fitbit_to_daily(entries, value_key="value", agg="mean"):
    """Aggregate minute/entry-level Fitbit data to daily values."""
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get(value_key, entry.get("value", None))
        if date and val is not None:
            try:
                val = float(val)
                rows.setdefault(date, []).append(val)
            except (ValueError, TypeError):
                pass
    if agg == "mean":
        return {d: np.mean(v) for d,v in rows.items()}
    elif agg == "sum":
        return {d: np.sum(v) for d,v in rows.items()}
    elif agg == "max":
        return {d: np.max(v) for d,v in rows.items()}
    return rows

#  Helper: load resting heart rate JSON
def load_resting_hr(path):
    """resting_heart_rate.json has nested value.value structure."""
    entries = load_fitbit_json(path)
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get("value",{})
        if isinstance(val, dict):
            rhr = val.get("value", val.get("restingHeartRate", None))
        else:
            rhr = val
        if date and rhr is not None:
            try:
                rows[date] = float(rhr)
            except (ValueError, TypeError):
                pass
    return rows

#  Helper: load sleep.json
def load_sleep(path):
    """Extract sleep efficiency and minutes asleep from sleep.json."""
    entries = load_fitbit_json(path)
    eff_rows = {}; min_rows = {}
    for entry in entries:
        # Each entry is one sleep session
        date = str(entry.get("dateOfSleep", entry.get("dateTime","")))[:10]
        eff  = entry.get("efficiency", None)
        mins = entry.get("minutesAsleep", None)
        if date and eff is not None:
            try:
                eff_rows[date] = float(eff)
            except (ValueError, TypeError):
                pass
        if date and mins is not None:
            try:
                min_rows[date] = float(mins)
            except (ValueError, TypeError):
                pass
    return eff_rows, min_rows

#  Helper: load sleep_score.csv
def load_sleep_score(path):
    """sleep_score.csv: timestamp, overall_score, composition_score, ..."""
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        # Timestamp column
        ts_col = next((c for c in df.columns if "timestamp" in c or "date" in c), None)
        score_col = next((c for c in df.columns if "overall" in c or "score" in c), None)
        if ts_col and score_col:
            df["date"] = pd.to_datetime(df[ts_col]).dt.strftime("%Y-%m-%d")
            return dict(zip(df["date"], pd.to_numeric(df[score_col], errors="coerce")))
    except Exception:
        pass
    return {}

#  Helper: load wellness.csv
def load_wellness(path):
    """
    pmsys/wellness.csv: daily self-report.
    Columns typically: date, fatigue, mood, readiness, sleep_duration,
                       sleep_quality, stress, soreness
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        if "date" not in df.columns:
            return pd.DataFrame()
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
        return df
    except Exception:
        return pd.DataFrame()

#  Helper: load googledocs/reporting.csv
def load_reporting(path):
    """
    reporting.csv: date, timestamp, meals, weight, glasses_of_fluid, alcohol
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        date_col = next((c for c in df.columns if "date" in c), None)
        if date_col:
            df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")
            return df
    except Exception:
        pass
    return pd.DataFrame()

#  Main loader
def load_pmdata_participant(pid_dir):
    """
    Load all data for one PMData participant.
    Returns a daily DataFrame with all available signals.
    """
    fitbit_dir    = os.path.join(pid_dir, "fitbit")
    pmsys_dir     = os.path.join(pid_dir, "pmsys")
    googledocs_dir= os.path.join(pid_dir, "googledocs")

    daily = {}  # date -> feature dict

    def add_series(series_dict, col_name):
        for date, val in series_dict.items():
            daily.setdefault(date, {})[col_name] = val

    # Resting heart rate (most reliable HR signal for daily level)
    rhr_path = os.path.join(fitbit_dir, "resting_heart_rate.json")
    if os.path.isfile(rhr_path):
        add_series(load_resting_hr(rhr_path), "resting_hr")

    # Steps
    steps_path = os.path.join(fitbit_dir, "steps.json")
    if os.path.isfile(steps_path):
        add_series(fitbit_to_daily(load_fitbit_json(steps_path), agg="sum"), "steps")

    # Activity minutes
    for fname, col in [
        ("lightly_active_minutes.json",    "lightly_active_min"),
        ("moderately_active_minutes.json", "moderately_active_min"),
        ("very_active_minutes.json",       "very_active_min"),
        ("sedentary_minutes.json",         "sedentary_min"),
    ]:
        p = os.path.join(fitbit_dir, fname)
        if os.path.isfile(p):
            add_series(fitbit_to_daily(load_fitbit_json(p), agg="sum"), col)

    # Sleep
    sleep_path = os.path.join(fitbit_dir, "sleep.json")
    if os.path.isfile(sleep_path):
        eff, mins = load_sleep(sleep_path)
        add_series(eff,  "sleep_efficiency")
        add_series(mins, "sleep_minutes")

    # Sleep score
    ss_path = os.path.join(fitbit_dir, "sleep_score.csv")
    if os.path.isfile(ss_path):
        add_series(load_sleep_score(ss_path), "sleep_score")

    if not daily:
        return pd.DataFrame()

    # Build base daily DataFrame
    df = pd.DataFrame.from_dict(daily, orient="index")
    df.index.name = "date"
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # Active minutes total
    act_cols = [c for c in ["lightly_active_min","moderately_active_min","very_active_min"] if c in df.columns]
    if act_cols:
        df["active_minutes_total"] = df[act_cols].sum(axis=1)

    # v1.4: wellness is NOT merged into days_df (v1.4 memo, Section 4). It is built as a separate
    # table under rules W1-W4 in cell W-1 and joined only when the Task 3 target is built.

    # Merge reporting (weight, fluid)
    r_path = os.path.join(googledocs_dir, "reporting.csv")
    if os.path.isfile(r_path):
        rep = load_reporting(r_path)
        if not rep.empty:
            rep["date"] = pd.to_datetime(rep["date"])
            rep_agg = rep.groupby("date").agg(
                weight         = ("weight", "mean"),
                glasses_of_fluid=("glasses_of_fluid","mean") if "glasses_of_fluid" in rep.columns else ("date","count"),
            ).reset_index()
            df = df.merge(rep_agg, on="date", how="left")

    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df


#  Load all participants
pmdata_participants = []
participant_dirs = sorted([
    os.path.join(PMDATA_DIR, d)
    for d in os.listdir(PMDATA_DIR)
    if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p")
])
print(f"Found {len(participant_dirs)} participant directories")

for pdir in participant_dirs:
    pid = os.path.basename(pdir)
    try:
        daily = load_pmdata_participant(pdir)
        if len(daily) < 7:
            print(f"  SKIP {pid}: only {len(daily)} days")
            continue

        # Get max HR from overview
        max_hr = np.nan
        if not overview.empty:
            hr_col = next((c for c in overview.columns if "heart" in c and "max" in c), None)
            id_col = next((c for c in overview.columns if "participant" in c or "id" in c), None)
            if hr_col and id_col:
                row = overview[overview[id_col].astype(str).str.lower().str.strip() == pid.lower()]
                if not row.empty:
                    max_hr = pd.to_numeric(row.iloc[0][hr_col], errors="coerce")   # v1.4: p12's cell is ' ' (v1.4 memo, Sec. 2); not a model feature

        pmdata_participants.append(dict(
            pid       = pid,
            group     = "pmdata",
            is_depressed = 0,
            max_hr    = max_hr,
            n_days    = len(daily),
            days_df   = daily,
        ))
        cols_found = [c for c in daily.columns if c != "date"]
        print(f"  OK {pid}: {len(daily)} days  max_HR={max_hr:.0f}  signals={cols_found[:6]}")
    except Exception as e:
        print(f"  ERROR {pid}: {e}")

print(f"\nPMData: {len(pmdata_participants)} participants loaded")
if pmdata_participants:
    print(f"  Median days: {np.median([p['n_days'] for p in pmdata_participants]):.0f}")
    all_cols = set()
    for p in pmdata_participants:
        all_cols.update(p['days_df'].columns.tolist())
    print(f"  All available columns: {sorted(all_cols)}")

In [ ]:
# ================================================================
# CELL 6  PMData FEATURE ENGINEERING + WELLNESS MERGE   [v1.4: wellness merge removed, marked 'v1.4']
# Defines feature set, builds neutrosophic features, and crucially
# merges wellness scores as the external target column.
# ================================================================
import numpy as np
import pandas as pd

if not pmdata_participants:
    print("No PMData participants loaded. Check PMDATA_DIR.")
    pmdata_all = pd.DataFrame()
    PMDATA_FEATURE_NAMES = []
    PMDATA_DOMAIN_MAP    = {}
    PMDATA_WELLNESS      = []
else:
    # Collect all columns across participants
    sample_cols = set()
    for p in pmdata_participants:
        sample_cols.update(p["days_df"].columns.tolist())
    sample_cols.discard("date")

    PMDATA_PHYS = [c for c in [
        "resting_hr","steps","active_minutes_total",
        "lightly_active_min","moderately_active_min",
        "very_active_min","sedentary_min"
    ] if c in sample_cols]

    PMDATA_SLEEP = [c for c in [
        "sleep_efficiency","sleep_minutes","sleep_score"
    ] if c in sample_cols]

    # Wellness columns — these become the EXTERNAL TARGET for lambda search
    PMDATA_WELLNESS = [c for c in [
        "fatigue","mood","readiness","sleep_quality","stress","soreness"
    ] if c in sample_cols]

    PMDATA_FEATURE_NAMES = PMDATA_PHYS + PMDATA_SLEEP
    PMDATA_DOMAIN_MAP    = {"physiology": PMDATA_PHYS, "sleep": PMDATA_SLEEP}

    print("PMData physiology features:", PMDATA_PHYS)
    print("PMData sleep features:     ", PMDATA_SLEEP)
    print("PMData wellness/target cols:", PMDATA_WELLNESS)
    print("Total NDI features:        ", len(PMDATA_FEATURE_NAMES))

    # v1.4: the wellness re-load and merge that stood here (UTC date, one row per REPORT) is removed: it is what
    # duplicated 18 days in v1-v1.3 (v1.4 memo). days_df stays one row per Fitbit date; wellness is built in W-1.
    print("v1.4: wellness kept out of days_df (separate table, cell W-1)")

    # Build neutrosophic features for each participant
    print("\nComputing neutrosophic features for PMData participants...")
    pmdata_frames = []
    for p in pmdata_participants:
        try:
            df = p["days_df"].copy().reset_index(drop=True)

            for col in PMDATA_FEATURE_NAMES:
                if col not in df.columns:
                    df[col] = np.nan
                df[col] = pd.to_numeric(df[col], errors="coerce")
                df[col] = df[col].fillna(df[col].median())

            X        = df[PMDATA_FEATURE_NAMES].values
            train_mask_full = np.ones(len(X), dtype=bool)
            neutro   = ndi.build_neutrosophic_features(X, PMDATA_FEATURE_NAMES,
                                            train_mask=train_mask_full)
            result   = pd.concat([df[["date"]].reset_index(drop=True),
                                   neutro.reset_index(drop=True)], axis=1)
            result["pid"]    = p["pid"]
            result["max_hr"] = p["max_hr"]

            # Keep raw features AND wellness columns
            for col in PMDATA_FEATURE_NAMES + PMDATA_WELLNESS:
                if col in df.columns:
                    result[col] = df[col].values

            pmdata_frames.append(result)
            p["neutro_df"] = result

            well_coverage = {c: df[c].notna().sum() for c in PMDATA_WELLNESS
                             if c in df.columns}
            print(f"  {p['pid']}: {len(result)} days  "
                  f"NDI={result['NDI'].mean():.4f}  "
                  f"wellness={well_coverage}")

        except Exception as e:
            print(f"  ERROR {p['pid']}: {e}")

    if pmdata_frames:
        pmdata_all = pd.concat(pmdata_frames, ignore_index=True)
        print(f"\nPMData total rows: {len(pmdata_all)}")
        print(f"NDI range: [{pmdata_all['NDI'].min():.4f},"
              f"{pmdata_all['NDI'].max():.4f}]")
        for wcol in PMDATA_WELLNESS:
            if wcol in pmdata_all.columns:
                vals = pmdata_all[wcol].dropna()
                if len(vals) > 0:
                    print(f"  {wcol}: mean={vals.mean():.2f}  "
                          f"range=[{vals.min():.0f},{vals.max():.0f}]  "
                          f"n={len(vals)}")
    else:
        print("No PMData frames built.")
        pmdata_all           = pd.DataFrame()
        PMDATA_FEATURE_NAMES = []
        PMDATA_DOMAIN_MAP    = {}
        PMDATA_WELLNESS      = []

In [ ]:
# D-2 (v1.4) One row per local calendar date: every days_df must have unique, consecutive dates (v1.4 memo, Section 4)
# days_df comes straight from the Fitbit daily files (CELL 5); wellness is not merged into it any more (CELL 5, CELL 6).
# Any duplicated, unparseable or missing date stops the run here, with the offending dates printed.
import numpy as np, pandas as pd
_WELL_COLS = {"fatigue", "mood", "readiness", "sleep_quality", "stress", "soreness", "effective_time_frame"}
_rows, _bad = [], []
for p in pmdata_participants:
    d = pd.to_datetime(p["days_df"]["date"], errors="coerce")
    dd = d.dropna()
    steps = dd.diff().dt.days.dropna()
    gaps = [(str(a.date()), str(b.date())) for a, b, s in zip(dd.iloc[:-1], dd.iloc[1:], steps) if s != 1]
    leaked = sorted(_WELL_COLS & set(p["days_df"].columns))
    _rows.append(dict(pid=p["pid"], rows=len(d), first_date=str(dd.min().date()), last_date=str(dd.max().date()),
                      unparseable_dates=int(d.isna().sum()), duplicated_dates=int(d.duplicated().sum()),
                      non_consecutive_steps=len(gaps), wellness_columns_in_days_df=len(leaked)))
    if d.isna().any() or d.duplicated().any() or gaps or leaked:
        _bad.append((p["pid"], int(d.isna().sum()), int(d.duplicated().sum()), gaps[:10], leaked))
DATE_CHECK = pd.DataFrame(_rows)
print(DATE_CHECK.to_string(index=False))
if _bad:
    for pid, n_nat, n_dup, g, lk in _bad:
        print(f"  {pid}: unparseable {n_nat}, duplicated {n_dup}, steps != 1 day (previous, next): {g}, wellness columns: {lk}")
    raise RuntimeError("STOP: days_df is not one row per consecutive calendar date (v1.4 memo, Section 4). "
                       "Send the lines above to ma'am; do not fix by hand.")
print(f"D-2 PASSED: {len(DATE_CHECK)} participants, {int(DATE_CHECK['rows'].sum())} rows, "
      "every date unique and consecutive, no wellness column in days_df")

In [ ]:
# W-1 (v1.4) Wellness table under W1-W4, kept OUT of days_df (v1.4 memo, Sections 3-4)
# One row per participant and local date; joined only when the Task 3 target is built. Tasks 1-2 do not use it.
# Same functions as W-0 in NDI_v1_4_prechecks.ipynb. Built here, early, so a problem stops the run in minutes;
# the files are written by X-1 into results_v1.4.
import os
import numpy as np, pandas as pd

# ---- Wellness rules W1-W4 (v1.4 memo, Section 3). Identical in NDI_v1_4_prechecks.ipynb (W-0) and NDI_v1_4.ipynb (W-1).
WELLNESS_TZ = "Europe/Oslo"
W_ITEMS_1_5 = ["mood", "stress", "fatigue", "sleep_quality", "soreness"]   # scored 1-5: 0 = missing (W2)
W_ITEMS_ALL = W_ITEMS_1_5 + ["readiness"]                                  # readiness runs 0-10: 0 is valid
W_RANGE = {**{c: (1, 5) for c in W_ITEMS_1_5}, "readiness": (0, 10)}

def load_wellness_reports(pmdata_dir, pid):
    """Every report in pmsys/wellness.csv, untouched, with its UTC and local (W1) calendar date."""
    path = os.path.join(pmdata_dir, pid, "pmsys", "wellness.csv")
    if not os.path.isfile(path):
        return None
    w = pd.read_csv(path)
    w.columns = [c.strip().lower().replace(" ", "_") for c in w.columns]
    raw = w["effective_time_frame"].astype(str).str.strip()
    ts = pd.to_datetime(raw, errors="coerce", utc=True)
    rep = pd.DataFrame({"pid": pid, "timestamp": raw.values, "ts_utc": ts.values})
    rep["ts_utc"] = pd.to_datetime(rep["ts_utc"], utc=True)
    rep["tz_aware"] = raw.str.contains(r"(?:Z|[+-]\d\d:?\d\d)$", regex=True).values
    rep["utc_date"] = rep["ts_utc"].dt.strftime("%Y-%m-%d")
    rep["local_date"] = rep["ts_utc"].dt.tz_convert(WELLNESS_TZ).dt.strftime("%Y-%m-%d")      # W1
    for c in W_ITEMS_ALL:
        rep[c] = pd.to_numeric(w[c], errors="coerce").values if c in w.columns else np.nan
    return rep

def apply_wellness_rules(rep):
    """W1-W4 for one participant. Returns (daily table: one row per local date, log dict)."""
    pid = rep["pid"].iloc[0] if len(rep) else None
    bad_ts = rep["ts_utc"].isna() | ~rep["tz_aware"]
    r = rep[~bad_ts].copy()
    log = dict(pid=pid, reports=len(rep), timestamps_unparseable_or_without_timezone=int(bad_ts.sum()),
               reports_moved_by_W1=int((r["utc_date"] != r["local_date"]).sum()))
    for c in W_ITEMS_ALL:                                   # anything other than 0 outside the scale
        lo, hi = W_RANGE[c]
        v = r[c]
        log[f"out_of_range_{c}"] = int((v.notna() & (v != 0) & ((v < lo) | (v > hi) | (v != v.round()))).sum())
    for c in W_ITEMS_1_5:
        log[f"zeros_{c}"] = int((r[c] == 0).sum())
    blank = (r[W_ITEMS_1_5] == 0).all(axis=1)              # W3: all five 1-5 items are 0 -> blank submission
    log["reports_dropped_W3"] = int(blank.sum())
    r = r[~blank].copy()
    for c in W_ITEMS_1_5:                                   # W2: 0 = missing on the 1-5 items
        log[f"zeros_set_missing_W2_{c}"] = int((r[c] == 0).sum())
        r.loc[r[c] == 0, c] = np.nan
    r = r.sort_values("ts_utc", kind="mergesort")
    g = r.groupby("local_date", sort=True)
    daily = g[W_ITEMS_ALL].last()                           # W4: last valid value of the day, item by item
    daily.insert(0, "reports_that_day", g.size())
    daily = daily.reset_index().rename(columns={"local_date": "date"})
    daily.insert(0, "pid", pid)
    log["local_days_with_a_report"] = len(daily)
    log["days_with_2plus_reports_W4"] = int((daily["reports_that_day"] >= 2).sum())
    for c in W_ITEMS_ALL:
        nvalid = g[c].count()
        nuniq = g[c].nunique()
        log[f"W4_days_choosing_{c}"] = int((nvalid >= 2).sum())         # days where W4 picked among valid values
        log[f"W4_days_values_differ_{c}"] = int((nuniq >= 2).sum())     # ... and the values disagreed
    return daily, log

def build_wellness_tables(pmdata_dir, pids):
    tables, logs = {}, []
    for pid in pids:
        rep = load_wellness_reports(pmdata_dir, pid)
        if rep is None or rep.empty:
            logs.append(dict(pid=pid, reports=0))
            continue
        tables[pid], lg = apply_wellness_rules(rep)
        logs.append(lg)
    log = pd.DataFrame(logs).fillna(0)
    for c in log.columns:
        if c != "pid":
            log[c] = log[c].astype(int)
    return tables, log

def wellness_zero_table(log):
    """Deliverable 2: zeros per participant and 1-5 item (all reports), blank reports W3 drops, zero share."""
    z = log[["pid", "reports"] + [f"zeros_{c}" for c in W_ITEMS_1_5] + ["reports_dropped_W3"]].copy()
    z["zeros_total"] = z[[f"zeros_{c}" for c in W_ITEMS_1_5]].sum(axis=1)
    z["zero_share_of_item_values"] = (z["zeros_total"] / (5 * z["reports"]).where(z["reports"] > 0)).round(3)
    z["zeros_left_after_W3"] = z["zeros_total"] - 5 * z["reports_dropped_W3"]
    return z

_pids = sorted(d for d in os.listdir(PMDATA_DIR) if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p"))
wellness_tables, wellness_log = build_wellness_tables(PMDATA_DIR, _pids)
wellness_daily = pd.concat(list(wellness_tables.values()), ignore_index=True) if wellness_tables else pd.DataFrame()
wellness_zeros = wellness_zero_table(wellness_log)
_chk = ["timestamps_unparseable_or_without_timezone"] + [f"out_of_range_{c}" for c in W_ITEMS_ALL]
_bad = wellness_log[wellness_log[_chk].sum(axis=1) > 0]
if not _bad.empty:
    print(_bad[["pid"] + _chk].to_string(index=False))
    raise RuntimeError("STOP: wellness timestamps or values the W1-W4 rules do not cover (see above). Report to ma'am.")
assert not wellness_daily.duplicated(["pid", "date"]).any(), "wellness table is not one row per local date"
_fit_dates = {p["pid"]: set(pd.to_datetime(p["days_df"]["date"]).dt.strftime("%Y-%m-%d")) for p in pmdata_participants}
wellness_log["local_days_outside_fitbit_dates"] = [
    int((~wellness_daily.loc[wellness_daily["pid"] == pid, "date"].isin(_fit_dates[pid])).sum()) if pid in _fit_dates else np.nan
    for pid in wellness_log["pid"]]
print(f"W-1: {len(wellness_daily)} participant-days from {int(wellness_log['reports'].sum())} reports | "
      f"W1 moved {int(wellness_log['reports_moved_by_W1'].sum())} reports | W3 dropped {int(wellness_log['reports_dropped_W3'].sum())} | "
      f"W2 set {int(wellness_zeros['zeros_left_after_W3'].sum())} zeros to missing | "
      f"days with 2+ reports (W4): {int(wellness_log['days_with_2plus_reports_W4'].sum())}, "
      f"mood values differing on {int(wellness_log['W4_days_values_differ_mood'].sum())} of them")

In [ ]:
# ================================================================
# CELL 6.5  F_SCALE PRE-SPECIFICATION SWEEP  (follow-up note, "The fix")
# Run this BEFORE Cell 7 and BEFORE inspecting any CV/rho result.
#
# Pre-specified rule (fixed in advance; do not revise after seeing
# the numbers): sweep SCALE over {1, 2, 3, 4, 5}. For each candidate,
# rebuild F for every feature and participant in both datasets, pool
# all rows, and compute per dataset:
#   F_IQR       = interquartile range of pooled F  (higher = less
#                 saturated / more graded)
#   |Sp(T,F)|   = |Spearman(T, F)| on pooled T/F
# Choose the SCALE that MAXIMISES the mean F_IQR across datasets,
# subject to the WORST-CASE (max across datasets) |Sp(T,F)| staying
# < 0.7. If no candidate satisfies the constraint on both datasets,
# keep the current CONFIG["F_SCALE"] and flag for manual review
# rather than silently picking a value that fails the independence
# check on one dataset.
# ================================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

SCALE_GRID = [1, 2, 3, 4, 5]
_original_f_scale = ndi.CONFIG["F_SCALE"]
sweep_rows = []

for scale in SCALE_GRID:
    ndi.CONFIG["F_SCALE"] = scale   # neutrosophic_decompose reads CONFIG at call time

    for dataset_name, participants, feat_names in [
        ("DEPRESJON", depresjon_participants, FEAT_NAMES),
        ("PMData",    pmdata_participants,    PMDATA_FEATURE_NAMES),
    ]:
        if not participants:
            continue
        T_pool, F_pool = [], []
        for p in participants:
            df = p["days_df"].copy().reset_index(drop=True)
            for col in feat_names:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                    df[col] = df[col].fillna(df[col].median())
                else:
                    df[col] = 0.0
            X = df[feat_names].values
            n = len(X)
            train_mask_full = np.ones(n, dtype=bool)  # same in-sample mode as Cell 4/6
            neutro_df = ndi.build_neutrosophic_features(
                X, feat_names, train_mask=train_mask_full)
            T_pool.append(neutro_df["Depression_T"].values)
            F_pool.append(neutro_df["Depression_F"].values)

        T_all = np.concatenate(T_pool)
        F_all = np.concatenate(F_pool)
        valid = ~(np.isnan(T_all) | np.isnan(F_all))
        T_all, F_all = T_all[valid], F_all[valid]

        f_iqr = float(np.percentile(F_all, 75) - np.percentile(F_all, 25))
        sp, _ = spearmanr(T_all, F_all)

        sweep_rows.append(dict(scale=scale, dataset=dataset_name,
                                F_IQR=f_iqr, abs_Sp_TF=abs(sp)))

sweep_df = pd.DataFrame(sweep_rows)
print("=" * 65)
print("F_SCALE PRE-SPECIFICATION SWEEP")
print("=" * 65)
print(sweep_df.to_string(index=False))

agg = (sweep_df.groupby("scale")
       .agg(mean_F_IQR=("F_IQR", "mean"),
            max_abs_Sp_TF=("abs_Sp_TF", "max"))
       .reset_index())
agg["passes_constraint"] = agg["max_abs_Sp_TF"] < 0.7
print()
print(agg.to_string(index=False))

qualifying = agg[agg["passes_constraint"]]
if len(qualifying) > 0:
    best_row = qualifying.loc[qualifying["mean_F_IQR"].idxmax()]
    CHOSEN_SCALE = float(best_row["scale"])
    print(f"\nCHOSEN F_SCALE = {CHOSEN_SCALE}  "
          f"(mean F_IQR={best_row['mean_F_IQR']:.3f}, "
          f"worst-case |Sp(T,F)|={best_row['max_abs_Sp_TF']:.3f} < 0.7)")
else:
    CHOSEN_SCALE = _original_f_scale
    print(f"\nNo SCALE in {SCALE_GRID} keeps |Sp(T,F)| < 0.7 on BOTH "
          f"datasets -- keeping current CONFIG['F_SCALE']="
          f"{_original_f_scale} and flagging for manual review "
          f"(widen the grid, or revisit the normaliser).")

# FREEZE. This is the only place F_SCALE should be set after this
# point. Chosen by the rule above, before any CV/SHAP/lambda result
# was inspected -- do not hand-tune this later after seeing rho.
ndi.CONFIG["F_SCALE"] = CHOSEN_SCALE
print(f"\nndi.CONFIG['F_SCALE'] frozen at {CHOSEN_SCALE}.")
if CHOSEN_SCALE != _original_f_scale:
    print("This differs from the value already baked into "
          "depresjon_all / pmdata_all (built in Cell 4 / Cell 6) -- "
          "RE-RUN those two cells now so Cell 7's verification "
          "reflects the frozen value. Per-fold CV cells (Component 2 "
          "onward) rebuild features from raw data on every call, so "
          "they pick up CONFIG['F_SCALE'] automatically.")
else:
    print("Matches the value already used in Cell 4 / Cell 6 -- no "
          "rebuild needed.")

In [ ]:
# ================================================================
# CELL 14.4  V2' — POPULATION VOLATILITY NORM + FOLD-WISE FRAME
# (V2' spec Sec 4) Population statistic is now the SPREAD OF PER-
# PERSON VOLATILITIES across donors, not raw values. Leakage rule
# unchanged: p never enters its own donor pool.
# ================================================================
def _compute_population_volatility_norm(participants, held_out_pid, feat_names, has_labels):
    donors = [q for q in participants if q["pid"] != held_out_pid]
    if has_labels:
        donors = [q for q in donors if q.get("group") == "control"]
    donor_ids = [q["pid"] for q in donors]

    assert held_out_pid not in donor_ids, \
        f"LEAKAGE: {held_out_pid} is in its own donor pool"

    donor_vols = []
    for q in donors:
        dfq = q["days_df"]
        vols = []
        for col in feat_names:
            v = pd.to_numeric(dfq[col], errors="coerce") if col in dfq.columns \
                else pd.Series(np.nan, index=dfq.index)
            vols.append(v.std())
        donor_vols.append(vols)
    donor_vols = np.array(donor_vols, dtype=float)  # (n_donors, d)

    pop_vol_median = np.nanmedian(donor_vols, axis=0)
    pop_vol_mad    = np.nanmedian(np.abs(donor_vols - pop_vol_median), axis=0) + 1e-8
    return pop_vol_median, pop_vol_mad, donor_ids


def build_v2prime_diagnostic_frame_foldwise(participants, feat_names, has_labels):
    """Fold-wise (not whole-series) -- V2's degenerate-frame lesson
    applies here too: F must vary within a participant to test
    independence against."""
    frames = []
    for p in participants:
        df_raw = p.get("days_df")
        if df_raw is None:
            continue
        df_raw = df_raw.copy().reset_index(drop=True)
        for col in feat_names:
            if col in df_raw.columns:
                df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
                df_raw[col] = df_raw[col].fillna(df_raw[col].median())
            else:
                df_raw[col] = 0.0
        n = len(df_raw)
        if n < 15:
            continue
        X_raw = df_raw[feat_names].values
        t0 = min(10, max(5, n // 3))

        pop_vol_median, pop_vol_mad, _ = _compute_population_volatility_norm(
            participants, p["pid"], feat_names, has_labels)

        try:
            fold_info = ndi.blocked_tscv(
                X_raw, np.zeros(n), ndi.rf_factory, T0=t0, return_folds=True)
            folds = fold_info.get("folds", [])
        except Exception:
            continue

        rows = []
        for fold in folds:
            tr_idx, te_idx = fold
            tr_idx = np.array(tr_idx); te_idx = np.array(te_idx)
            if int(te_idx.min()) - int(tr_idx.max()) < 6:
                continue
            train_mask = np.zeros(n, dtype=bool)
            train_mask[tr_idx] = True

            neutro_fold = ndi.build_neutrosophic_features(
                X_raw, feat_names, train_mask=train_mask,
                f_mode="population_vol",
                pop_vol_median=pop_vol_median, pop_vol_mad=pop_vol_mad)
            rows.append(neutro_fold.iloc[te_idx])

        if not rows:
            continue
        pf = pd.concat(rows, ignore_index=True)
        pf["pid"]   = p["pid"]
        pf["group"] = p.get("group", "pmdata")
        frames.append(pf)

    return pd.concat(frames, ignore_index=True)

In [ ]:
# ================================================================
# CELL 14.10  ITEM 2a -- V3_BEST_LAMBDA (frozen; run completed fully
# before this session disconnected -- both datasets printed, no
# need to rerun the 12hr search)
#
#   PMData:    best_lambda=(0.20, 0.15, 0.65)  mean_rho=0.3834
#              original_mean_rho=0.3198  n_stable=13  Robust
#   DEPRESJON: best_lambda=(0.00, 1.00, 0.00)  mean_rho=0.6325
#              original_mean_rho=0.2962  n_stable=1
#              NOT robust -- corner solution, near-zero stability
#              radius. Not used downstream (V3_BEST_LAMBDA freezes
#              PMData's), kept here only for the write-up's caveat.
# ================================================================
V3_BEST_LAMBDA = (0.20, 0.15, 0.65)

pmdata_lambda_result = dict(
    best_lambda=V3_BEST_LAMBDA, best_mean_rho=0.3834,
    original_mean_rho=0.3198, n_stable=13,
    verdict="Robust — weights stable across neighbourhood")

depresjon_lambda_result = dict(
    best_lambda=(0.0, 1.0, 0.0), best_mean_rho=0.6325,
    original_mean_rho=0.2962, n_stable=1,
    verdict="Per-subject tuning recommended (small stability radius) — corner solution, not robust")

print(f"V3_BEST_LAMBDA (frozen from PMData primary calibration): {V3_BEST_LAMBDA}")

## Repository (tag v1.4-task3) and Stage 0 settings

In [ ]:
# G-0 (Task 3) Run from inside the repository so T3-0 can see git; same as in NDI_v1_4.ipynb except RUN_TAG
# Works with the private repository: if a plain clone/fetch fails it asks once for a read-only token
# (typed into a hidden prompt, never stored in the notebook or in .git/config), then checks out the
# tag so the run's HEAD is the tagged commit. Idempotent.
import os, subprocess, getpass
REPO_URL = "https://github.com/hpatel09112004/NDI_Depression_Risk_Index.git"
REPO_DIR = "/content/NDI_Depression_Risk_Index"
RUN_TAG = "v1.4-task3"
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")

def _git_run(args, secret=None):
    r = subprocess.run(args, capture_output=True, text=True, env=_env)
    msg = (r.stderr or "")[-600:]
    return r.returncode, (msg.replace(secret, "***") if secret else msg)

def _token_url():
    tok = getpass.getpass("Private repository: paste the read-only GitHub token (hidden, not saved): ").strip()
    return REPO_URL.replace("https://", f"https://hpatel09112004:{tok}@"), tok

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _rc, _msg = _git_run(["git", "clone", "--quiet", REPO_URL, REPO_DIR])
    if _rc != 0:
        _url, _tok = _token_url()
        _rc, _msg = _git_run(["git", "clone", "--quiet", _url, REPO_DIR], secret=_tok)
        assert _rc == 0, "git clone failed (token or URL?):\n" + _msg
        _git_run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])   # token not kept on disk
else:
    _rc, _msg = _git_run(["git", "-C", REPO_DIR, "fetch", "--tags", "--quiet"])
    _have = _git_run(["git", "-C", REPO_DIR, "rev-parse", "-q", "--verify", f"refs/tags/{RUN_TAG}"])[0] == 0
    if _rc != 0 and not _have:
        _url, _tok = _token_url()
        _rc, _msg = _git_run(["git", "-C", REPO_DIR, "fetch", "--tags", "--quiet", _url], secret=_tok)
        assert _rc == 0, "git fetch failed (token or URL?):\n" + _msg
os.chdir(REPO_DIR)
if _git_run(["git", "rev-parse", "-q", "--verify", f"refs/tags/{RUN_TAG}"])[0] == 0:
    _rc, _msg = _git_run(["git", "checkout", "--quiet", "--force", RUN_TAG])   # --force: a config file left by an untagged attempt is replaced by the tagged one
    assert _rc == 0, f"could not check out {RUN_TAG}:\n" + _msg
    print(f"Checked out tag {RUN_TAG} (detached HEAD at the tagged commit)")
else:
    print(f"Tag {RUN_TAG} not found yet: T3-0 will write the config and stop.")
print("Working directory:", os.getcwd())
print("HEAD:", subprocess.run(["git", "log", "-1", "--format=%H %cI %s"], capture_output=True, text=True).stdout.strip())
print("Tags:", subprocess.run(["git", "tag", "--list"], capture_output=True, text=True).stdout.split())

In [ ]:
# S0-0 Stage 0 settings and provenance (Task 3 review / Task 4 memo, Section 2). Read-only on every archive.
# Analysis settings: v1.4 (tag v1.4) and Task 3 (tag v1.4-task3), checked against the tags; nothing is re-tuned.
# Writes only to results_task4_stage0/.
import os, json, time, subprocess, datetime, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon, bootstrap as scipy_bootstrap
from sklearn.linear_model import Ridge

_REQUIRED = ["ndi", "pmdata_participants", "PMDATA_FEATURE_NAMES", "RANDOM_SEED", "RESULTS_DIR", "PMDATA_DIR",
             "_compute_population_volatility_norm", "DATE_CHECK", "wellness_daily", "wellness_log"]
_missing = [g for g in _REQUIRED if g not in globals()]
assert not _missing, f"Run the notebook from the top first. Missing: {_missing}"

def _git(*args):
    try:
        return subprocess.check_output(["git", *args], stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return None

_BASE_DIR = os.path.dirname(RESULTS_DIR)
RESULTS_V1_4_ARCHIVE = os.path.join(_BASE_DIR, "results_v1.4")           # read-only
RESULTS_T3_ARCHIVE = os.path.join(_BASE_DIR, "results_v1.4_task3")       # read-only
RESULTS_S0_DIR = os.path.join(_BASE_DIR, "results_task4_stage0")
_need = {RESULTS_V1_4_ARCHIVE: ["config_v1_4.json", "table1_corrected_self_target_per_participant.csv",
                                "raw_target_corrected_per_participant.csv", "wellness_daily_W1_W4.csv"],
         RESULTS_T3_ARCHIVE: ["config_task3_v1_4.json", "task3_mood_per_participant.csv",
                              "task3_stress_per_participant.csv", "persistence_reference_per_participant.csv",
                              "persistence_reference_summary.csv"]}
_absent = [os.path.join(os.path.basename(d), f) for d, fs in _need.items() for f in fs if not os.path.isfile(os.path.join(d, f))]
assert not _absent, f"archived files needed by Stage 0 are missing: {_absent}"

V1_2_CONFIG = json.load(open(os.path.join(RESULTS_V1_4_ARCHIVE, "config_v1_4.json")))   # name kept for V12-1/V12-2
_t = _git("show", "v1.4:config_v1_4.json")
assert _t is not None and json.loads(_t) == V1_2_CONFIG, "results_v1.4/config_v1_4.json differs from tag v1.4 -- STOP"
TASK3_CONFIG = json.load(open(os.path.join(RESULTS_T3_ARCHIVE, "config_task3_v1_4.json")))
_t = _git("show", "v1.4-task3:config_task3_v1_4.json")
assert _t is not None and json.loads(_t) == TASK3_CONFIG, "results_v1.4_task3/config_task3_v1_4.json differs from tag v1.4-task3 -- STOP"
assert float(ndi.CONFIG["F_SCALE"]) == V1_2_CONFIG["F_SCALE_FROZEN"], "F_SCALE is not the frozen 1.0 -- STOP"

STAGE0_CONFIG = dict(
    VERSION="v1.4-task4-stage0", MEMO="Task 3 review and Task 4 execution memo, Section 2",
    STATUS="diagnostic, exploratory; never enters any reported comparison",
    ARM="learned persistence: personalised Z's model (fold z-scoring, then Ridge with RIDGE_ALPHA = %s) given one "
        "feature, today's value of the target quantity" % ndi.CONFIG["RIDGE_ALPHA"],
    PROTOCOL="the exact v1.4 / Task 3 fold protocol: V12-2 get_folds (T0 rule, embargo g = 6, fold size 1), the "
             "embargo check, purge_training_rows with the target's alignment, training rows and test days with a "
             "finite target, >= 5 training rows; the code is run_arms' fold loop with one arm",
    FEATURE=dict(self_derived="today's causal index value (the index build_target returns, fixed lambda (0.2, 0.3, 0.5))",
                 raw_feature="today's causal raw composite (the quantity the raw-feature target smooths; T3-P (c))",
                 mood_stress="today's W1-W4 value; a training row or test day without it is dropped"),
    PARTICIPANTS=dict(self_and_raw="the v1.4 Table 1 / raw-feature participants (results_v1.4 per-participant files)",
                      mood_and_stress="the Task 3 eligible participants of each target"),
    REFERENCE="no-model persistence: Spearman(today's value, target) on the SAME days the learned arm scores; it must "
              "equal the archived Task 3 persistence values (T3-P) on the Task 3 days",
    DECISION="PROCEED if, for every target, mean learned persistence >= 0.5 x mean no-model persistence on the same "
             "days; otherwise STOP and send the per-participant values (memo, Section 2)",
    WELLNESS_USE="wellness values are a feature only for the mood / stress rows of this diagnostic",
    OUTPUT="results_task4_stage0/")

os.makedirs(RESULTS_S0_DIR, exist_ok=True)
with open(os.path.join(RESULTS_S0_DIR, "stage0_config.json"), "w") as f:
    json.dump(STAGE0_CONFIG, f, indent=2)
PROVENANCE_S0 = dict(run_started=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                     working_directory=os.getcwd(), head_commit=_git("rev-parse", "HEAD"),
                     tag_v1_4_task3_commit=_git("rev-list", "-n", "1", "v1.4-task3"),
                     v1_4_config_matches_tag=True, task3_config_matches_tag=True, f_scale=float(ndi.CONFIG["F_SCALE"]),
                     pmdata_participants_loaded=[p["pid"] for p in pmdata_participants])
with open(os.path.join(RESULTS_S0_DIR, "provenance.json"), "w") as f:
    json.dump(PROVENANCE_S0, f, indent=2)
print(json.dumps(PROVENANCE_S0, indent=2))

SEED, N_BOOT, SESOI = V1_2_CONFIG["BOOTSTRAP_SEED"], V1_2_CONFIG["BOOTSTRAP_N"], V1_2_CONFIG["SESOI"]
K, G, MF = V1_2_CONFIG["SMOOTH_K"], V1_2_CONFIG["EMBARGO_G"], V1_2_CONFIG["MIN_FOLDS"]
ARMS = ["vsigned", "v3", "rawrf", "persz"]
ARM_LABEL = {"vsigned": "V-signed", "v3": "V3", "rawrf": "Raw-feature RF", "persz": "Personalised Z"}
print(f"Stage 0 settings checked against tags v1.4 and v1.4-task3; outputs -> {RESULTS_S0_DIR}")

## v1.4 code (V12-1, V12-2) and Task 3's next-day targets (T3-1), unchanged

In [ ]:
# V12-1 (v1.3: forward branch of smooth_target uses min_periods=k) Building blocks: imputation, causal targets, target support, purge
def _feature_matrix_nan(d0, feat_names):
    """Raw feature matrix with NaN kept (columns absent from the frame -> 0.0, as in v1)."""
    n = len(d0)
    cols = [pd.to_numeric(d0[c], errors="coerce").values if c in d0.columns else np.zeros(n)
            for c in feat_names]
    return np.column_stack(cols).astype(float)

def impute_from_rows(X_nan, row_mask):
    """Fill NaN with the column median over row_mask only (column all-NaN there -> 0.0)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(X_nan[row_mask], axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    return np.where(np.isnan(X_nan), med, X_nan)

def impute_whole_series_v1(d0, feat_names):
    """Exact v1 behaviour: whole-series median per column."""
    df = d0.copy()
    for col in feat_names:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = 0.0
    return df[feat_names].values

def impute_causal(X_nan, B):
    """Row d filled with the median of rows [0, max(d+1, B)). Causal after the burn-in."""
    n = len(X_nan); out = X_nan.copy()
    for d in range(n):
        if np.isnan(X_nan[d]).any():
            L = min(max(d + 1, B), n)
            m = np.zeros(n, bool); m[:L] = True
            out[d] = impute_from_rows(X_nan, m)[d]
    return out

def smooth_target(idx, alignment, k=K):
    s = pd.Series(np.asarray(idx, float))
    if alignment == "trailing_v1":      # y[s] = mean(idx[s-k .. s-1])  (v1 convention)
        return s.rolling(k, min_periods=1).mean().shift(1).values
    if alignment == "forward":          # y[s] = mean(idx[s+1 .. s+k])
        # v1.3: full window required (min_periods=k). Rows with fewer than k following rows get NaN
        # and are dropped from scoring; with min_periods=1 they averaged fewer than 7 days.
        return s[::-1].rolling(k, min_periods=k).mean()[::-1].shift(-1).values
    raise ValueError(alignment)

def target_support(s, alignment, n, k=K):
    """Days whose index values enter y[s]."""
    if alignment == "trailing_v1":
        return set(range(max(0, s - k), s))
    return set(range(s + 1, min(n, s + k + 1)))

def purge_training_rows(tr_idx, te_idx, alignment, n):
    """Drop training rows whose target window touches the test day or its target window,
    or that lie inside the test day's target window."""
    test_days = set(int(t) for t in te_idx)
    test_support = set().union(*[target_support(int(s), alignment, n) for s in te_idx])
    blocked = test_days | test_support
    keep = [int(t) for t in tr_idx
            if int(t) not in test_support and not (target_support(int(t), alignment, n) & blocked)]
    return np.array(keep, dtype=int), len(tr_idx) - len(keep)

In [ ]:
# V12-2 Participant preparation, causal channels, the unified runner
_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE = {}, {}, {}
RUN_LOGS = {}   # label -> per-participant fold log (burn-in used, rows purged, first fold days)

def prepare(p, feat_names, has_labels, participants):
    key = (p["pid"], tuple(feat_names), bool(has_labels))
    if key in _PREP_CACHE:
        return _PREP_CACHE[key]
    d0 = p["days_df"].copy().reset_index(drop=True)
    n = len(d0)
    X_nan = _feature_matrix_nan(d0, feat_names)
    X_ws = impute_whole_series_v1(d0, feat_names)
    pm, pmad, _ = _compute_population_volatility_norm(participants, p["pid"], feat_names, has_labels)
    full = np.ones(n, bool)
    T_ws = ndi.compute_T_v3(X_ws, full)
    mi, ii = ndi.compute_quality_signal(d0, feat_names, full)
    I_ws = ndi.compute_I_v3(mi, ii)
    F_ws = ndi.compute_F_v3(X_ws, feat_names, full, pm, pmad)
    out = dict(pid=p["pid"], d0=d0, n=n, X_nan=X_nan, X_ws=X_ws, pm=pm, pmad=pmad,
               T_ws=T_ws, I_ws=I_ws, F_ws=F_ws, feat_names=list(feat_names),
               t0=min(10, max(5, n // 3)))
    _PREP_CACHE[key] = out
    return out

def get_folds(pr, g):
    fi = ndi.blocked_tscv(pr["X_ws"], np.zeros(pr["n"]), ndi.rf_factory,
                          T0=pr["t0"], g=g, return_folds=True)
    return [(np.array(a, int), np.array(b, int)) for a, b in fi.get("folds", [])]

def causal_channels(pr, B):
    """T, I, F where day d is computed from rows [0, max(d+1, B)) only."""
    key = (pr["pid"], tuple(pr["feat_names"]), B)
    if key in _CAUSAL_CACHE:
        return _CAUSAL_CACHE[key]
    n, fn = pr["n"], pr["feat_names"]
    Xc = impute_causal(pr["X_nan"], B)
    T, I, F = (np.full(n, np.nan) for _ in range(3))
    for d in range(n):
        L = min(max(d + 1, B), n); ones = np.ones(L, bool)
        T[d] = ndi.compute_T_v3(Xc[:L], ones)[d]
        F[d] = ndi.compute_F_v3(Xc[:L], fn, ones, pr["pm"], pr["pmad"])[d]
        mi, ii = ndi.compute_quality_signal(pr["d0"].iloc[:L].reset_index(drop=True), fn, ones)
        I[d] = ndi.compute_I_v3(mi, ii)[d]
    _CAUSAL_CACHE[key] = (T, I, F, Xc)
    return _CAUSAL_CACHE[key]

def build_target(pr, target, target_stats, alignment, lam, B):
    """Returns (y, index_series_or_None)."""
    n = pr["n"]
    if target_stats == "whole_series_v1":
        if target == "self":
            idx = ndi.compute_ndi_v3_primary(pr["T_ws"], pr["I_ws"], pr["F_ws"], lambda_vec=tuple(lam))
            y = pd.Series(idx).rolling(K, min_periods=1).mean().shift(1).fillna(0).values
            return y, np.asarray(idx, float)
        X = pr["X_ws"]; comp = ((X - X.mean(0)) / (X.std(0) + 1e-8)).mean(1)
        return pd.Series(comp).rolling(K, min_periods=1).mean().shift(1).values, None
    T, I, F, Xc = causal_channels(pr, B)
    if target == "self":
        idx = np.asarray(ndi.compute_ndi_v3_primary(T, I, F, lambda_vec=tuple(lam)), float)
        return smooth_target(idx, alignment), idx
    comp = np.full(n, np.nan)
    for d in range(n):
        L = min(max(d + 1, B), n)
        mu, sd = Xc[:L].mean(0), Xc[:L].std(0) + 1e-8
        comp[d] = ((Xc[d] - mu) / sd).mean()
    return smooth_target(comp, alignment), None

def fold_features(pr, tr_idx, te_idx, whole_series, need_vsigned):
    fn = pr["feat_names"]
    if whole_series:                                  # v1 feature path (= leak L1)
        n = pr["n"]; tm = np.zeros(n, bool); tm[tr_idx] = True
        X = pr["X_ws"]
        X3 = np.column_stack([pr["T_ws"], pr["I_ws"], pr["F_ws"]])
    else:                                             # corrected: training stats, truncated at test day
        L = int(te_idx.max()) + 1
        tm = np.zeros(L, bool); tm[tr_idx] = True
        X = impute_from_rows(pr["X_nan"][:L], tm)
        T = ndi.compute_T_v3(X, tm)
        mi, ii = ndi.compute_quality_signal(pr["d0"].iloc[:L].reset_index(drop=True), fn, tm)
        I = ndi.compute_I_v3(mi, ii)
        F = ndi.compute_F_v3(X, fn, tm, pr["pm"], pr["pmad"])
        X3 = np.column_stack([T, I, F])
    XVS = None
    if need_vsigned:
        vs = ndi.build_neutrosophic_features(X, fn, train_mask=tm)
        XVS = vs[["Depression_T", "Depression_I", "Depression_F"]].fillna(0).values
    return X, X3, XVS

def _zscore_ridge(X_tr, y_tr, X_te):
    mu = X_tr.mean(0); sg = X_tr.std(0) + 1e-8
    return Ridge(alpha=ndi.CONFIG["RIDGE_ALPHA"]).fit((X_tr - mu) / sg, y_tr).predict((X_te - mu) / sg)

def run_arms(participants, feat_names, has_labels, target="self", arms=ARMS,
             target_stats="causal", alignment=None, whole_series_features=False,
             g=G, purge=None, lam=None, burn_in=None, target_mode="fixed", label=""):
    """One runner for Table 1, the raw-target comparison and every Task 2 state.
    target: "self" | "raw".  target_mode: "fixed" (primary) | "per_fold" (sensitivity row)."""
    alignment = alignment or V1_2_CONFIG["TARGET_ALIGNMENT"]
    purge = V1_2_CONFIG["PURGE_SUPPORT_OVERLAP"] if purge is None else purge
    lam = tuple(lam or V1_2_CONFIG["FIXED_LAMBDA"])
    burn_in = burn_in or V1_2_CONFIG["TARGET_BURN_IN_DAYS"]
    v1_mode = (target_stats == "whole_series_v1")
    rows, log, t_start = [], [], time.time()
    for p in participants:
        if p.get("days_df") is None:
            continue
        pr = prepare(p, feat_names, has_labels, participants)
        n = pr["n"]
        if n < 15:
            continue
        try:
            folds = get_folds(pr, g)
        except Exception:
            continue
        if not folds:
            continue
        B = min(burn_in, int(min(int(tr.max()) for tr, _ in folds)) + 1)   # burn-in inside every training window
        key = (pr["pid"], tuple(feat_names), target, target_stats, alignment, whole_series_features,
               g, purge, lam, B, target_mode)
        cached = _RUN_CACHE.get(key)
        if cached is not None and set(arms) <= set(cached["preds"]):
            res = cached
        else:
            y_fixed, idx = build_target(pr, target, "whole_series_v1" if v1_mode else "causal",
                                        alignment, lam, B)
            preds = {a: [] for a in arms}; y_all, te_all, n_purged = [], [], 0
            for tr_idx, te_idx in folds:
                if int(te_idx.min()) - int(tr_idx.max()) < g:
                    continue
                if purge and not v1_mode:
                    tr_idx, npg = purge_training_rows(tr_idx, te_idx, alignment, n); n_purged += npg
                Xf, X3, XVS = fold_features(pr, tr_idx, te_idx, whole_series_features or v1_mode,
                                            need_vsigned="vsigned" in arms)
                if target_mode == "per_fold":
                    assert target == "self" and alignment == "trailing_v1", "per-fold rebuild: trailing self-target only"
                    idx_f = ndi.compute_ndi_v3_primary(X3[:, 0], X3[:, 1], X3[:, 2], lambda_vec=lam)
                    y = smooth_target(idx_f, alignment)
                else:
                    y = y_fixed
                tr_v = tr_idx[np.isfinite(y[tr_idx])]; te_v = te_idx[np.isfinite(y[te_idx])]
                if len(tr_v) < 5 or len(te_v) < 1:
                    continue
                try:
                    fold_pred = {}
                    if "vsigned" in arms:
                        m = ndi.rf_factory(); m.fit(XVS[tr_v], y[tr_v]); fold_pred["vsigned"] = m.predict(XVS[te_v])
                    if "v3" in arms:
                        m = ndi.rf_factory(); m.fit(X3[tr_v], y[tr_v]); fold_pred["v3"] = m.predict(X3[te_v])
                    if "rawrf" in arms:
                        m = ndi.rf_factory(); m.fit(Xf[tr_v], y[tr_v]); fold_pred["rawrf"] = m.predict(Xf[te_v])
                    if "persz" in arms:
                        fold_pred["persz"] = _zscore_ridge(Xf[tr_v], y[tr_v], Xf[te_v])
                except Exception:
                    continue
                for a in arms:
                    preds[a].extend(np.asarray(fold_pred[a], float).tolist())
                y_all.extend(y[te_v].tolist()); te_all.extend(te_v.tolist())
            res = dict(preds=preds, y=y_all, te=te_all, idx=idx, y_fixed=y_fixed, B=B, n_purged=n_purged)
            _RUN_CACHE[key] = res
        if len(res["y"]) < 3:
            continue
        yt = np.array(res["y"]); row = dict(pid=pr["pid"], n_folds=len(yt))
        for a in arms:
            pa = np.array(res["preds"][a]); ok = np.isfinite(yt) & np.isfinite(pa)
            row[a] = spearmanr(yt[ok], pa[ok])[0] if ok.sum() >= 3 else np.nan
        rows.append(row)
        log.append(dict(pid=pr["pid"], burn_in_used=res["B"], purged_rows_summed_over_folds=res["n_purged"],
                        first_train_max=int(folds[0][0].max()), first_test_day=int(folds[0][1].min())))
    out = pd.DataFrame(rows)
    RUN_LOGS[label or target] = pd.DataFrame(log)
    print(f"  [{label or target}] {len(out)} participants, {time.time() - t_start:.0f}s")
    return out

def l2_proxy_rho(participants, feat_names, has_labels, lam, alignment=None, g=G, burn_in=None):
    """Leak L2: no model -- target index at the most recent available day vs the target,
    on exactly the test days the model-based runner evaluated."""
    alignment = alignment or V1_2_CONFIG["TARGET_ALIGNMENT"]
    burn_in = burn_in or V1_2_CONFIG["TARGET_BURN_IN_DAYS"]
    rows = []
    for p in participants:
        if p.get("days_df") is None:
            continue
        pr = prepare(p, feat_names, has_labels, participants)
        if pr["n"] < 15:
            continue
        folds = get_folds(pr, g)
        if not folds:
            continue
        B = min(burn_in, int(min(int(tr.max()) for tr, _ in folds)) + 1)
        y, idx = build_target(pr, "self", "causal", alignment, tuple(lam), B)
        days = np.array(sorted({int(s) for _, te in folds for s in te}))
        lag = 1 if alignment == "trailing_v1" else 0
        days = days[days >= lag]
        sc, tg = idx[days - lag], y[days]
        ok = np.isfinite(sc) & np.isfinite(tg)
        if ok.sum() >= 3:
            rows.append(dict(pid=pr["pid"], n_folds=int(ok.sum()), v3=spearmanr(sc[ok], tg[ok])[0]))
    return pd.DataFrame(rows)

In [ ]:
# T3-1 Task 3 targets behind flags: target "mood" / "stress" with alignment "next_day".
# build_target and target_support are wrapped, not edited: every v1.4 target and alignment goes to the v1.4 code
# unchanged (checked in T3-R). run_arms, folds, purge, features, arms and models are the V12-2 code as run in v1.4.
_need = ["build_target", "target_support", "run_arms", "wellness_daily", "load_wellness_reports", "W_ITEMS_1_5",
         "DATE_CHECK", "TASK3_CONFIG"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run the cells above first. Missing: {_miss}"
TASK3_TARGETS = ("mood", "stress")
if "_build_target_v14" not in globals():            # wrap once, even if this cell is re-run
    _build_target_v14, _target_support_v14 = build_target, target_support

def target_support(s, alignment, n, k=K):
    """v1.4 rule for 'trailing_v1' / 'forward'; 'next_day': row s is scored against day s + 1 only."""
    if alignment == "next_day":
        return {s + 1}
    return _target_support_v14(s, alignment, n, k)

def build_target(pr, target, target_stats, alignment, lam, B):
    """Task 3 targets: the raw W1-W4 value of the next calendar day (NaN if there is none). Others: v1.4 code."""
    if target in TASK3_TARGETS:
        assert alignment == "next_day", "Task 3 targets are next-day values: call run_arms(..., alignment='next_day')"
        return TASK3_Y[(pr["pid"], target)].copy(), None
    return _build_target_v14(pr, target, target_stats, alignment, lam, B)

# --- targets: y[s] = value on local day (date of row s) + 1, from the W1-W4 table; rows are consecutive days (D-2)
_wd = wellness_daily.copy(); _wd["date"] = pd.to_datetime(_wd["date"])
TASK3_Y = {}
for p in pmdata_participants:
    d = pd.to_datetime(p["days_df"]["date"]).reset_index(drop=True)
    w = _wd[_wd["pid"] == p["pid"]].set_index("date")
    for t in TASK3_TARGETS:
        TASK3_Y[(p["pid"], t)] = (d + pd.Timedelta(days=1)).map(w[t]).astype(float).values

# --- feature check: no wellness.csv field enters any arm's features
_well_fields = set()
for _pid in sorted(x for x in os.listdir(PMDATA_DIR) if x.startswith("p")):
    _wp = os.path.join(PMDATA_DIR, _pid, "pmsys", "wellness.csv")
    if os.path.isfile(_wp):
        _well_fields |= {c.strip().lower().replace(" ", "_") for c in pd.read_csv(_wp, nrows=0).columns}
assert not (set(PMDATA_FEATURE_NAMES) & _well_fields), "a wellness.csv field is a model feature -- STOP"
assert (DATE_CHECK["wellness_columns_in_days_df"] == 0).all(), "wellness columns in days_df -- STOP"
FEATURE_CHECK = (f"No wellness.csv field enters any arm's features: the {len(PMDATA_FEATURE_NAMES)} features "
                 f"({', '.join(PMDATA_FEATURE_NAMES)}) come from the Fitbit files only; wellness.csv fields "
                 f"({', '.join(sorted(_well_fields))}) are not in days_df (D-2) and are used only to build the "
                 "Task 3 target.")
print(FEATURE_CHECK)

# --- per-day W4 information for the log: valid values of each item that local day, after W3 and W2
W4_DAY = {}
for p in pmdata_participants:
    rep = load_wellness_reports(PMDATA_DIR, p["pid"])
    if rep is None or rep.empty:
        continue
    r = rep[rep["ts_utc"].notna() & rep["tz_aware"]].copy()
    r = r[~(r[W_ITEMS_1_5] == 0).all(axis=1)].copy()                    # W3
    for c in W_ITEMS_1_5:
        r.loc[r[c] == 0, c] = np.nan                                    # W2
    g = r.groupby("local_date")
    for t in TASK3_TARGETS:
        W4_DAY[(p["pid"], t)] = (g[t].count(), g[t].nunique())
for t in TASK3_TARGETS:
    n_days = sum(int(np.isfinite(v).sum()) for (pid, tt), v in TASK3_Y.items() if tt == t)
    print(f"{t}: {n_days} participant-rows have a next-day value")

## Stage 0

In [ ]:
# S0-1 Statistics helpers, copied unchanged from T3-3 (= V12-6 of v1.4): BCa bootstrap, formatting.
def _rng():
    return np.random.default_rng(SEED)

def _boot(x, stat, level):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 3 or np.allclose(x, x[0]):
        return np.nan, np.nan, "degenerate"
    for method in ("BCa", "percentile"):
        try:
            r = scipy_bootstrap((x,), stat, n_resamples=N_BOOT, method=method,
                                confidence_level=level, vectorized=False, rng=_rng())
        except TypeError:   # scipy < 1.15
            r = scipy_bootstrap((x,), stat, n_resamples=N_BOOT, method=method,
                                confidence_level=level, vectorized=False, random_state=_rng())
        lo, hi = float(r.confidence_interval.low), float(r.confidence_interval.high)
        if np.isfinite(lo) and np.isfinite(hi):
            return lo, hi, method
    return np.nan, np.nan, "failed"

_ND = TASK3_CONFIG["REPORTING_DECIMALS"]
def _f(x):
    return "n/a" if pd.isna(x) else f"{x:.{_ND}f}"
def _ci(m, l, h):
    return f"{_f(m)} [{_f(l)}, {_f(h)}]"

In [ ]:
# S0-2 Learned persistence on the real data (Task 3 review / Task 4 memo, Section 2). Diagnostic, exploratory.
# One arm, personalised Z's model (fold z-scoring + Ridge, V12-2 _zscore_ridge) on ONE feature: today's value of the
# target quantity. The fold loop below is run_arms' loop (V12-2) with that single arm: same folds (get_folds), embargo
# check, purge (target's alignment), finite-target rows, >= 5 training rows. Checks, per participant: the arm scores the
# same number of test days as the archived run; no-model persistence on the Task 3 days equals the archived T3-P value.
# About 15 minutes on Colab (causal channels for the self-derived and raw-feature targets).
_need = ["prepare", "get_folds", "build_target", "purge_training_rows", "causal_channels", "smooth_target",
         "_zscore_ridge", "TASK3_Y", "wellness_daily", "_boot"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run the cells above first. Missing: {_miss}"
S0_LAM = tuple(V1_2_CONFIG["FIXED_LAMBDA"])
_FOLDS = {}
_wdS = wellness_daily.copy(); _wdS["date"] = pd.to_datetime(_wdS["date"])

def _folds_for(pr):
    if pr["pid"] not in _FOLDS:
        _FOLDS[pr["pid"]] = get_folds(pr, G)
    return _FOLDS[pr["pid"]]

def _today_value(p, pr, target, B, idx, alignment):
    """Today's value of the target quantity for every row (NaN where it does not exist)."""
    if target == "self":
        return np.asarray(idx, float)
    if target == "raw":                                   # T3-P (c): today's causal raw composite
        n = pr["n"]; _, _, _, Xc = causal_channels(pr, B)
        comp = np.full(n, np.nan)
        for d in range(n):
            L = min(max(d + 1, B), n)
            mu, sd = Xc[:L].mean(0), Xc[:L].std(0) + 1e-8
            comp[d] = ((Xc[d] - mu) / sd).mean()
        y_ref, _ = build_target(pr, "raw", "causal", alignment, S0_LAM, B)
        assert np.array_equal(np.nan_to_num(smooth_target(comp, alignment), nan=-9e9), np.nan_to_num(y_ref, nan=-9e9)), \
            f"{p['pid']}: today's composite does not rebuild the raw-feature target -- STOP"
        return comp
    d = pd.to_datetime(p["days_df"]["date"]).reset_index(drop=True)   # mood / stress: today's W1-W4 value (T3-P (a))
    return d.map(_wdS[_wdS["pid"] == p["pid"]].set_index("date")[target]).astype(float).values

def learned_persistence(p, target, alignment):
    pr = prepare(p, PMDATA_FEATURE_NAMES, False, pmdata_participants); n = pr["n"]
    folds = _folds_for(pr)
    B = min(V1_2_CONFIG["TARGET_BURN_IN_DAYS"], int(min(int(tr.max()) for tr, _ in folds)) + 1)
    y, idx = build_target(pr, target, "causal", alignment, S0_LAM, B)
    x = _today_value(p, pr, target, B, idx, alignment)
    scored, used, pred, n_purged = [], [], [], 0
    for tr_idx, te_idx in folds:
        if int(te_idx.min()) - int(tr_idx.max()) < G:
            continue
        tr_idx, npg = purge_training_rows(tr_idx, te_idx, alignment, n); n_purged += npg
        tr_v = tr_idx[np.isfinite(y[tr_idx])]; te_v = te_idx[np.isfinite(y[te_idx])]
        if len(tr_v) < 5 or len(te_v) < 1:
            continue
        scored.extend(te_v.tolist())                                       # the days every arm scores
        tr_f = tr_v[np.isfinite(x[tr_v])]; te_f = te_v[np.isfinite(x[te_v])]   # no value today -> row dropped
        if len(tr_f) < 5 or len(te_f) < 1:
            continue
        pred.extend(np.asarray(_zscore_ridge(x[tr_f][:, None], y[tr_f], x[te_f][:, None]), float).tolist())
        used.extend(te_f.tolist())
    scored, used, pred = np.array(scored, int), np.array(used, int), np.array(pred, float)
    t3_days = scored[np.isfinite(x[scored])]
    def _sp(a, b):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return float(spearmanr(a, b)[0]) if len(a) >= 3 else np.nan
    return dict(pid=p["pid"], scored_test_days=len(scored), days_with_today_value=len(t3_days), days_learned=len(used),
                purged_rows_summed_over_folds=n_purged, learned_rho=_sp(y[used], pred),
                nomodel_rho_same_days=_sp(x[used], y[used]), nomodel_rho_task3_days=_sp(x[t3_days], y[t3_days]))

_arch_pp = pd.read_csv(os.path.join(RESULTS_T3_ARCHIVE, "persistence_reference_per_participant.csv"))
_arch_sum = pd.read_csv(os.path.join(RESULTS_T3_ARCHIVE, "persistence_reference_summary.csv")).set_index("target")
_SPECS = [("self", V1_2_CONFIG["TARGET_ALIGNMENT"], "self-derived (v1.4)",
           pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "table1_corrected_self_target_per_participant.csv"))),
          ("raw", V1_2_CONFIG["TARGET_ALIGNMENT"], "raw-feature (v1.4)",
           pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "raw_target_corrected_per_participant.csv")))]
for _t in ("mood", "stress"):
    _a = pd.read_csv(os.path.join(RESULTS_T3_ARCHIVE, f"task3_{_t}_per_participant.csv"))
    _SPECS.append((_t, "next_day", f"next-day {_t}", _a[_a["eligible"].astype(str).str.lower() == "true"]))

_t0 = time.time(); _rows = []
for target, alignment, label, arms_df in _SPECS:
    ref = _arch_pp[_arch_pp["target"] == label].set_index("pid")
    for pid in arms_df["pid"]:
        p = next(q for q in pmdata_participants if q["pid"] == pid)
        r = learned_persistence(p, target, alignment)
        a = arms_df.set_index("pid").loc[pid]
        r.update(target=label, archived_scored_test_days=int(a["n_folds"]),
                 archived_days_used=int(ref.loc[pid, "days_used"]), archived_nomodel_rho=float(ref.loc[pid, "persistence_rho"]),
                 **{f"archived_{k}": float(a[k]) for k in ARMS})
        _rows.append(r)
    print(f"  {label}: {len(arms_df)} participants, {time.time() - _t0:.0f}s", flush=True)
S0_PP = pd.DataFrame(_rows)
_cols = ["target", "pid", "scored_test_days", "archived_scored_test_days", "days_with_today_value", "archived_days_used",
         "days_learned", "purged_rows_summed_over_folds", "learned_rho", "nomodel_rho_same_days", "nomodel_rho_task3_days",
         "archived_nomodel_rho"] + [f"archived_{k}" for k in ARMS]
S0_PP = S0_PP[_cols]

# --- protocol checks: same scored days as the archived runs; no-model persistence reproduces T3-P exactly
_same_days = (S0_PP["scored_test_days"] == S0_PP["archived_scored_test_days"]) & (S0_PP["days_with_today_value"] == S0_PP["archived_days_used"])
_a, _b = S0_PP["nomodel_rho_task3_days"].to_numpy(float), S0_PP["archived_nomodel_rho"].to_numpy(float)
_same_rho = (np.isnan(_a) & np.isnan(_b)) | (np.abs(_a - _b) < 1e-9)
if not (_same_days.all() and _same_rho.all()):
    print(S0_PP[~(_same_days & _same_rho)].to_string(index=False))
    raise RuntimeError("STOP: Stage 0 does not score the archived days or does not reproduce the archived persistence "
                       "values (rows above). Send them to ma'am.")
S0_PP.to_csv(os.path.join(RESULTS_S0_DIR, "stage0_learnability_per_participant.csv"), index=False)
print(f"Protocol checks passed: scored days equal the archived runs for {len(S0_PP)} participant-targets; no-model "
      f"persistence on the Task 3 days reproduces T3-P exactly ({time.time() - _t0:.0f}s)")

In [ ]:
# S0-3 Stage 0 table and decision (rule fixed in the memo, Section 2): PROCEED if, on every target, mean learned
# persistence >= 0.5 x mean no-model persistence on the same days; otherwise STOP and send the per-participant values.
def _mci(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    lo, hi, _ = _boot(v, np.mean, 0.95)
    return (float(v.mean()) if len(v) else np.nan), lo, hi, len(v)

_sum = []
for target, alignment, label, arms_df in _SPECS:
    g = S0_PP[S0_PP["target"] == label]
    both = g[np.isfinite(g["learned_rho"]) & np.isfinite(g["nomodel_rho_same_days"])]
    lm, llo, lhi, ln = _mci(both["learned_rho"])
    nm, nlo, nhi, nn = _mci(both["nomodel_rho_same_days"])
    dm, dlo, dhi, _ = _mci(both["learned_rho"] - both["nomodel_rho_same_days"])
    am = {k: float(arms_df[k].mean()) for k in ARMS}
    best = max(am, key=am.get)
    ratio = lm / nm if nm and np.isfinite(nm) else np.nan
    _sum.append(dict(target=label, participants=len(g), participants_with_both_rho=len(both),
                     days_learned=int(both["days_learned"].sum()),
                     learned_persistence_mean_rho=lm, learned_ci95_lo=llo, learned_ci95_hi=lhi,
                     nomodel_persistence_mean_rho_same_days=nm, nomodel_ci95_lo=nlo, nomodel_ci95_hi=nhi,
                     learned_minus_nomodel=dm, diff_ci95_lo=dlo, diff_ci95_hi=dhi, ratio_learned_to_nomodel=ratio,
                     archived_nomodel_mean_rho=float(_arch_sum["mean_rho"].get(label, np.nan)),
                     best_arm=ARM_LABEL[best], best_arm_mean_rho=am[best],
                     passes_half_rule=bool(np.isfinite(ratio) and lm >= 0.5 * nm)))
S0_SUM = pd.DataFrame(_sum)
S0_SUM.to_csv(os.path.join(RESULTS_S0_DIR, "stage0_learnability_summary.csv"), index=False)
S0_FMT = pd.DataFrame({
    "Target": S0_SUM["target"], "n": S0_SUM["participants_with_both_rho"], "Days": S0_SUM["days_learned"],
    "No-model persistence [95% CI]": [_ci(m, l, h) for m, l, h in zip(S0_SUM["nomodel_persistence_mean_rho_same_days"], S0_SUM["nomodel_ci95_lo"], S0_SUM["nomodel_ci95_hi"])],
    "Learned persistence [95% CI]": [_ci(m, l, h) for m, l, h in zip(S0_SUM["learned_persistence_mean_rho"], S0_SUM["learned_ci95_lo"], S0_SUM["learned_ci95_hi"])],
    "Learned - no-model [95% CI]": [_ci(m, l, h) for m, l, h in zip(S0_SUM["learned_minus_nomodel"], S0_SUM["diff_ci95_lo"], S0_SUM["diff_ci95_hi"])],
    "Ratio": [_f(v) for v in S0_SUM["ratio_learned_to_nomodel"]],
    "Best arm (mean rho)": [f"{a} ({_f(v)})" for a, v in zip(S0_SUM["best_arm"], S0_SUM["best_arm_mean_rho"])],
    ">= half": ["yes" if v else "NO" for v in S0_SUM["passes_half_rule"]]})
S0_FMT.to_csv(os.path.join(RESULTS_S0_DIR, "stage0_learnability_summary_formatted.csv"), index=False)
pd.set_option("display.width", 250)
print(S0_FMT.to_string(index=False))

DECISION = "PROCEED" if S0_SUM["passes_half_rule"].all() else "STOP"
_fail = S0_SUM[~S0_SUM["passes_half_rule"]]["target"].tolist()
if DECISION == "PROCEED":
    reading = ("Learned persistence comes close to the no-model value on every target (at least half of it): the pipeline "
               "learns a signal it is given, so the arms' low values reflect weak learning from 10 features and short "
               "histories. That is a finding to report. Proceed to Task 4.")
else:
    reading = (f"Learned persistence falls below half the no-model value on: {', '.join(_fail)}. A training-row alignment "
               "problem would affect Task 4 as well: STOP and send results_task4_stage0/stage0_learnability_per_participant.csv "
               "to ma'am (memo, Section 2).")
_dec = dict(decision=DECISION, rule=STAGE0_CONFIG["DECISION"], failing_targets=_fail, reading=reading,
            summary=S0_SUM.round(6).to_dict(orient="records"), finished=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))
with open(os.path.join(RESULTS_S0_DIR, "stage0_decision.json"), "w") as f:
    json.dump(_dec, f, indent=2, default=float)
notes = ["STAGE 0 LEARNABILITY CHECK (Task 3 review / Task 4 memo, Section 2) -- diagnostic, exploratory",
         "Arm: " + STAGE0_CONFIG["ARM"] + ". Protocol: " + STAGE0_CONFIG["PROTOCOL"] + ".",
         "Checks: every participant-target scores the same test days as the archived run; no-model persistence on the "
         "Task 3 days reproduces the archived T3-P values exactly.",
         "Learned and no-model persistence are compared on the same days (the days with today's value).",
         "Results: " + "; ".join(f"{t}: learned {l} vs no-model {n} (ratio {q})" for t, l, n, q in
                                 zip(S0_FMT["Target"], S0_FMT["Learned persistence [95% CI]"],
                                     S0_FMT["No-model persistence [95% CI]"], S0_FMT["Ratio"])),
         f"DECISION: {DECISION}. {reading}"]
with open(os.path.join(RESULTS_S0_DIR, "stage0_notes.txt"), "w") as f:
    f.write("\n".join(notes) + "\n")
print("\n" + "\n".join(notes[-2:]))
_files = ["stage0_config.json", "provenance.json", "stage0_learnability_per_participant.csv", "stage0_learnability_summary.csv",
          "stage0_learnability_summary_formatted.csv", "stage0_decision.json", "stage0_notes.txt"]
_m = [f for f in _files if not os.path.isfile(os.path.join(RESULTS_S0_DIR, f))]
print(f"\nSTAGE 0 FILES: {len(_files) - len(_m)} of {len(_files)} in {RESULTS_S0_DIR}" + (f"; MISSING {_m}" if _m else ""))
print(f"STAGE 0 DECISION: {DECISION}")

## Diagnostic after STOP (does not change the decision)

In [ ]:
# S0-4 Diagnostic after STOP (changes nothing above; writes two extra files to results_task4_stage0/).
# Question: is the loss a training-row alignment problem, or does refitting per fold bury a weak signal?
# Each learned prediction is  pred = a + b * z,  a = intercept (the training mean of y, which moves fold to fold),
# b = slope, z = today's value z-scored on that fold's training rows. Per participant it reports:
#   rho_learned   Spearman(pred, y)       -- must equal S0-2's learned_rho (checked)
#   rho_slope     Spearman(b * z, y)      -- the part that uses today's value
#   rho_intercept Spearman(a, y)          -- the part that only moves with the fold
#   share_b_pos   share of folds with a positive slope (alignment problem -> slopes near 0 or negative)
#   sd_a, sd_bz   spread of the two parts across test days
# Run it in the same session, right after S0-3 (about 2 minutes: everything is cached).
_need = ["S0_PP", "_SPECS", "_folds_for", "_today_value", "prepare", "build_target", "purge_training_rows", "_zscore_ridge"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run the notebook from the top first (Runtime > Run all). Missing: {_miss}"

def _sp4(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(spearmanr(a, b)[0]) if len(a) >= 3 and np.ptp(a) > 0 and np.ptp(b) > 0 else np.nan

def lp_parts(p, target, alignment):
    pr = prepare(p, PMDATA_FEATURE_NAMES, False, pmdata_participants); n = pr["n"]
    folds = _folds_for(pr)
    B = min(V1_2_CONFIG["TARGET_BURN_IN_DAYS"], int(min(int(tr.max()) for tr, _ in folds)) + 1)
    y, idx = build_target(pr, target, "causal", alignment, S0_LAM, B)
    x = _today_value(p, pr, target, B, idx, alignment)
    rec = []
    for tr_idx, te_idx in folds:
        if int(te_idx.min()) - int(tr_idx.max()) < G:
            continue
        tr_idx, _ = purge_training_rows(tr_idx, te_idx, alignment, n)
        tr_v = tr_idx[np.isfinite(y[tr_idx])]; te_v = te_idx[np.isfinite(y[te_idx])]
        if len(tr_v) < 5 or len(te_v) < 1:
            continue
        tr_f = tr_v[np.isfinite(x[tr_v])]; te_f = te_v[np.isfinite(x[te_v])]
        if len(tr_f) < 5 or len(te_f) < 1:
            continue
        mu, sg = x[tr_f].mean(), x[tr_f].std() + 1e-8                       # = _zscore_ridge's scaling
        m = Ridge(alpha=ndi.CONFIG["RIDGE_ALPHA"]).fit(((x[tr_f] - mu) / sg)[:, None], y[tr_f])
        z = (x[te_f] - mu) / sg
        pred = m.intercept_ + m.coef_[0] * z
        ref = np.asarray(_zscore_ridge(x[tr_f][:, None], y[tr_f], x[te_f][:, None]), float)
        assert np.allclose(pred, ref, atol=1e-10), f"{p['pid']}: decomposition does not reproduce _zscore_ridge"
        for s, zz, pp in zip(te_f, z, pred):
            rec.append((int(s), float(m.intercept_), float(m.coef_[0]), float(zz), float(pp), float(y[s]), float(x[s])))
    r = pd.DataFrame(rec, columns=["day", "a", "b", "z", "pred", "y", "x"])
    return dict(pid=p["pid"], days=len(r), rho_learned=_sp4(r["pred"], r["y"]), rho_slope=_sp4(r["b"] * r["z"], r["y"]),
                rho_intercept=_sp4(r["a"], r["y"]), rho_nomodel=_sp4(r["x"], r["y"]),
                share_b_pos=float((r["b"] > 0).mean()) if len(r) else np.nan,
                sd_a=float(r["a"].std()), sd_bz=float((r["b"] * r["z"]).std()))

_t0 = time.time(); _d = []
for target, alignment, label, arms_df in _SPECS:
    for pid in arms_df["pid"]:
        p = next(q for q in pmdata_participants if q["pid"] == pid)
        _d.append(dict(target=label, **lp_parts(p, target, alignment)))
S0_DIAG = pd.DataFrame(_d)
_chk = S0_DIAG.merge(S0_PP[["target", "pid", "learned_rho"]], on=["target", "pid"])
_a, _b = _chk["rho_learned"].to_numpy(float), _chk["learned_rho"].to_numpy(float)
assert ((np.isnan(_a) & np.isnan(_b)) | (np.abs(_a - _b) < 1e-9)).all(), "diagnostic does not reproduce S0-2 -- STOP"
S0_DIAG.to_csv(os.path.join(RESULTS_S0_DIR, "stage0_diagnostic_per_participant.csv"), index=False)
_g = (S0_DIAG.groupby("target", sort=False)
      .agg(n=("pid", "size"), no_model=("rho_nomodel", "mean"), learned=("rho_learned", "mean"),
           slope_part=("rho_slope", "mean"), intercept_part=("rho_intercept", "mean"),
           share_slope_positive=("share_b_pos", "mean"), median_sd_intercept=("sd_a", "median"),
           median_sd_slope_part=("sd_bz", "median")).reset_index())
_g.to_csv(os.path.join(RESULTS_S0_DIR, "stage0_diagnostic_summary.csv"), index=False)
pd.set_option("display.width", 250)
print(_g.round(3).to_string(index=False))
print(f"\n({time.time() - _t0:.0f}s; decomposition reproduces S0-2's learned rho exactly)")
print("Reading: if slope_part is close to no_model and share_slope_positive is high, the model learned the right "
      "relation on correctly aligned rows, and the loss comes from the intercept moving between folds "
      "(median_sd_intercept comparable to or larger than median_sd_slope_part). An alignment problem would show "
      "slope_part far below no_model and slopes near zero or negative.")

## Stage 0b: anomaly re-scoring (Stage 0 review memo, Section 1; diagnostic, exploratory, gates nothing)

In [ ]:
# S0-5 Stage 0b -- anomaly re-scoring (Stage 0 review memo, Section 1). DIAGNOSTIC, EXPLORATORY; gates nothing.
# For every scored test day s, a(s) = mean of the target over the training rows used for that day's fold (the Ridge
# intercept of learned persistence, S0-4). Score any forecast f by rho_anom = Spearman(f(s) - a(s), y(s) - a(s)), per
# participant over the same days as Stage 0, then averaged across participants as before.
#   learned persistence: f = the Ridge prediction;  no-model persistence: f = today's value x(s) (on the target's scale).
# Reading (fixed in the memo): learned >= 0.5 x no-model on all four targets -> refit drift explains the Stage 0 gap;
# otherwise the slopes themselves are unstable on weak targets. Either reading is reported. Writes stage0b_* only.
_need = ["S0_PP", "_SPECS", "_folds_for", "_today_value", "prepare", "build_target", "purge_training_rows",
         "_zscore_ridge", "_boot", "_ci", "_f"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run the notebook from the top first (Runtime > Run all). Missing: {_miss}"

def _sp5(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(spearmanr(a, b)[0]) if len(a) >= 3 and np.ptp(a) > 0 and np.ptp(b) > 0 else np.nan

def lp_days(p, target, alignment):
    """S0-2's fold loop, one row per scored day: a(s), learned prediction, target, today's value."""
    pr = prepare(p, PMDATA_FEATURE_NAMES, False, pmdata_participants); n = pr["n"]
    folds = _folds_for(pr)
    B = min(V1_2_CONFIG["TARGET_BURN_IN_DAYS"], int(min(int(tr.max()) for tr, _ in folds)) + 1)
    y, idx = build_target(pr, target, "causal", alignment, S0_LAM, B)
    x = _today_value(p, pr, target, B, idx, alignment)
    rec = []
    for tr_idx, te_idx in folds:
        if int(te_idx.min()) - int(tr_idx.max()) < G:
            continue
        tr_idx, _ = purge_training_rows(tr_idx, te_idx, alignment, n)
        tr_v = tr_idx[np.isfinite(y[tr_idx])]; te_v = te_idx[np.isfinite(y[te_idx])]
        if len(tr_v) < 5 or len(te_v) < 1:
            continue
        tr_f = tr_v[np.isfinite(x[tr_v])]; te_f = te_v[np.isfinite(x[te_v])]
        if len(tr_f) < 5 or len(te_f) < 1:
            continue
        pred = np.asarray(_zscore_ridge(x[tr_f][:, None], y[tr_f], x[te_f][:, None]), float)
        a = float(y[tr_f].mean())                                  # = the Ridge intercept (z-scored feature)
        for s, pp in zip(te_f, pred):
            rec.append((int(s), a, float(pp), float(y[s]), float(x[s])))
    return pd.DataFrame(rec, columns=["day", "a", "pred", "y", "x"])

_t0 = time.time(); _r5, _days5 = [], []
for target, alignment, label, arms_df in _SPECS:
    for pid in arms_df["pid"]:
        p = next(q for q in pmdata_participants if q["pid"] == pid)
        d = lp_days(p, target, alignment)
        _days5.append(d.assign(target=label, pid=pid))
        _r5.append(dict(target=label, pid=pid, days=len(d),
                        learned_rho=_sp5(d["pred"], d["y"]), nomodel_rho=_sp5(d["x"], d["y"]),
                        learned_rho_anom=_sp5(d["pred"] - d["a"], d["y"] - d["a"]),
                        nomodel_rho_anom=_sp5(d["x"] - d["a"], d["y"] - d["a"])))
S0B_PP = pd.DataFrame(_r5)
# the same days and values as Stage 0 (S0-2): ordinary rho reproduced exactly
_chk = S0B_PP.merge(S0_PP[["target", "pid", "learned_rho", "nomodel_rho_same_days", "days_learned"]],
                    on=["target", "pid"], suffixes=("", "_s0"))
def _eq(u, v):
    u, v = np.asarray(u, float), np.asarray(v, float)
    return bool(((np.isnan(u) & np.isnan(v)) | (np.abs(u - v) < 1e-9)).all())
assert (_chk["days"] == _chk["days_learned"]).all() and _eq(_chk["learned_rho"], _chk["learned_rho_s0"]) \
       and _eq(_chk["nomodel_rho"], _chk["nomodel_rho_same_days"]), "Stage 0b does not reproduce Stage 0's days or rho -- STOP"
S0B_PP.to_csv(os.path.join(RESULTS_S0_DIR, "stage0b_anomaly_per_participant.csv"), index=False)
pd.concat(_days5, ignore_index=True).to_csv(os.path.join(RESULTS_S0_DIR, "stage0b_anomaly_per_day.csv"), index=False)

_s5 = []
for target, alignment, label, arms_df in _SPECS:
    g = S0B_PP[S0B_PP["target"] == label]
    both = g[np.isfinite(g["learned_rho_anom"]) & np.isfinite(g["nomodel_rho_anom"])]
    lm, llo, lhi, _ = _mci(both["learned_rho_anom"]); nm, nlo, nhi, _ = _mci(both["nomodel_rho_anom"])
    _s5.append(dict(target=label, participants=len(g), participants_with_both=len(both), days=int(both["days"].sum()),
                    nomodel_anom_mean=nm, nomodel_anom_ci95_lo=nlo, nomodel_anom_ci95_hi=nhi,
                    learned_anom_mean=lm, learned_anom_ci95_lo=llo, learned_anom_ci95_hi=lhi,
                    ratio=lm / nm if nm and np.isfinite(nm) else np.nan,
                    learned_ge_half_nomodel=bool(np.isfinite(lm) and np.isfinite(nm) and lm >= 0.5 * nm),
                    stage0_nomodel_mean=float(both["nomodel_rho"].mean()), stage0_learned_mean=float(both["learned_rho"].mean())))
S0B_SUM = pd.DataFrame(_s5)
S0B_SUM.to_csv(os.path.join(RESULTS_S0_DIR, "stage0b_anomaly_summary.csv"), index=False)
S0B_FMT = pd.DataFrame({
    "Target": S0B_SUM["target"], "n": S0B_SUM["participants_with_both"], "Days": S0B_SUM["days"],
    "Stage 0 no-model / learned (rho)": [f"{_f(a)} / {_f(b)}" for a, b in zip(S0B_SUM["stage0_nomodel_mean"], S0B_SUM["stage0_learned_mean"])],
    "No-model, anomaly rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(S0B_SUM["nomodel_anom_mean"], S0B_SUM["nomodel_anom_ci95_lo"], S0B_SUM["nomodel_anom_ci95_hi"])],
    "Learned, anomaly rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(S0B_SUM["learned_anom_mean"], S0B_SUM["learned_anom_ci95_lo"], S0B_SUM["learned_anom_ci95_hi"])],
    "Ratio": [_f(v) for v in S0B_SUM["ratio"]],
    ">= half": ["yes" if v else "no" for v in S0B_SUM["learned_ge_half_nomodel"]]})
S0B_FMT.to_csv(os.path.join(RESULTS_S0_DIR, "stage0b_anomaly_summary_formatted.csv"), index=False)
pd.set_option("display.width", 250)
print(S0B_FMT.to_string(index=False))

_all = bool(S0B_SUM["learned_ge_half_nomodel"].all())
_fail5 = S0B_SUM[~S0B_SUM["learned_ge_half_nomodel"]]["target"].tolist()
READING_0B = ("Under anomaly scoring, learned persistence is at least half the no-model value on all four targets: refit "
              "drift (the intercept moving between folds) explains the Stage 0 gap." if _all else
              "Under anomaly scoring, learned persistence stays below half the no-model value on: " + ", ".join(_fail5) +
              ". Removing the refit drift does not close the gap there: the slopes themselves are unstable on weak targets.")
with open(os.path.join(RESULTS_S0_DIR, "stage0b_reading.json"), "w") as f:
    json.dump(dict(label="Stage 0b, diagnostic and exploratory (Stage 0 review memo, Section 1); gates nothing",
                   definition="a(s) = mean of the target over the training rows of that day's fold (the Ridge intercept); "
                              "rho_anom = Spearman(f - a, y - a) per participant over Stage 0's days, averaged",
                   rule="learned >= 0.5 x no-model (anomaly rho) on all four targets -> refit drift explains the gap; "
                        "otherwise the slopes are unstable on weak targets",
                   all_four_pass=_all, below_half=_fail5, reading=READING_0B,
                   summary=S0B_SUM.round(6).to_dict(orient="records"),
                   finished=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")), f, indent=2, default=float)
with open(os.path.join(RESULTS_S0_DIR, "stage0b_notes.txt"), "w") as f:
    f.write("STAGE 0b ANOMALY RE-SCORING -- diagnostic, exploratory; gates nothing (Stage 0 review memo, Section 1)\n"
            "a(s) = mean of the target over the training rows of the fold that scores day s (the Ridge intercept); "
            "rho_anom = Spearman(f(s) - a(s), y(s) - a(s)), per participant over the same days as Stage 0, averaged.\n"
            "Check: Stage 0's days and its ordinary learned / no-model rho are reproduced exactly.\n"
            + "; ".join(f"{t}: no-model {n} vs learned {l} (ratio {q})" for t, n, l, q in
                        zip(S0B_FMT["Target"], S0B_FMT["No-model, anomaly rho [95% CI]"],
                            S0B_FMT["Learned, anomaly rho [95% CI]"], S0B_FMT["Ratio"])) + "\n"
            + "READING: " + READING_0B + "\n")
print(f"\n({time.time() - _t0:.0f}s; Stage 0's days and rho reproduced exactly)")
print("STAGE 0b READING: " + READING_0B)